# Disease Prediction - XGBoost Training with GPU (T4)

This notebook trains an XGBoost model for disease prediction using One-Hot encoded symptom data with GPU acceleration.

**Prerequisites:**
- Runtime: GPU (T4)
- Dataset: `Final_Augmented_dataset_Diseases_and_Symptoms.csv`

In [ ]:
# Install required packages
!pip install xgboost scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

print("XGBoost version:", xgb.__version__)
print("GPU Available:", xgb.get_config())

## 1. Upload Dataset

Upload `Final_Augmented_dataset_Diseases_and_Symptoms.csv` using the Files panel on the left.

In [ ]:
# Load the dataset
print("📂 Loading dataset...")
df = pd.read_csv('Final_Augmented_dataset_Diseases_and_Symptoms.csv')
print(f"✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"✅ Diseases: {df['diseases'].nunique()} unique values")
df.head()

## 2. Prepare Data

Extract features (symptoms) and target (diseases). The dataset is already One-Hot encoded.

In [ ]:
# Separate features and target
print("🔄 Preparing data...")
X = df.drop(columns=['diseases'])
y = df['diseases']

# Get symptom list (feature names)
symptom_list = list(X.columns)
print(f"✅ Features (symptoms): {len(symptom_list)}")
print(f"✅ Target (diseases): {y.nunique()} unique classes")

In [ ]:
# Filter out classes with only 1 sample (stratify requires at least 2)
from collections import Counter
disease_counts = Counter(y)
valid_diseases = [disease for disease, count in disease_counts.items() if count >= 2]

# Filter dataset to only include diseases with at least 2 samples
mask = y.isin(valid_diseases)
X_filtered = X[mask]
y_filtered = y[mask]

print(f"⚠️  Filtered out {len(y) - len(y_filtered)} samples from {len(disease_counts) - len(valid_diseases)} diseases with <2 instances")
print(f"✅ Remaining diseases: {len(valid_diseases)}")

# Now encode the filtered diseases
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_filtered)
print(f"✅ Label encoding complete: {len(label_encoder.classes_)} classes")
print(f"   Example: '{label_encoder.classes_[0]}' → 0")
print(f"   Example: '{label_encoder.classes_[-1]}' → {len(label_encoder.classes_)-1}")

In [ ]:
# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"✅ Train set: {X_train.shape[0]} samples")
print(f"✅ Test set: {X_test.shape[0]} samples")

## 3. Train XGBoost Model with GPU

Using `tree_method='hist'` with `device='cuda'` for GPU acceleration on T4.

In [ ]:
print("🚀 Training XGBoost model with GPU...")

# XGBoost parameters optimized for GPU
params = {
    'objective': 'multi:softprob',
    'num_class': len(label_encoder.classes_),
    'tree_method': 'hist',
    'device': 'cuda',
    'max_depth': 10,
    'learning_rate': 0.1,
    'n_estimators': 200,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'eval_metric': 'mlogloss',
    'early_stopping_rounds': 20
}

# Create and train model
model = xgb.XGBClassifier(**params)

# Train with evaluation set
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=10
)

print("✅ Training complete!")

## 4. Evaluate Model Performance

In [ ]:
# Make predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
f1_weighted = f1_score(y_test, y_pred, average='weighted')
f1_macro = f1_score(y_test, y_pred, average='macro')

print("\n" + "="*60)
print("📊 MODEL PERFORMANCE METRICS")
print("="*60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1-Score (Weighted): {f1_weighted:.4f}")
print(f"F1-Score (Macro): {f1_macro:.4f}")
print("="*60)

In [ ]:
# Detailed classification report
print("\n📋 DETAILED CLASSIFICATION REPORT:\n")

# Get unique labels present in test set
unique_test_labels = np.unique(y_test)
target_names_filtered = [label_encoder.classes_[i] for i in unique_test_labels]

report = classification_report(
    y_test, 
    y_pred, 
    labels=unique_test_labels,
    target_names=target_names_filtered, 
    digits=3
)
print(report)

In [ ]:
# Show top 10 most common diseases in test set with their F1-scores
from collections import Counter

disease_counts = Counter([label_encoder.inverse_transform([y])[0] for y in y_test])
top_diseases = disease_counts.most_common(10)

print("\n🏥 TOP 10 DISEASES IN TEST SET:\n")
print(f"{'Disease':<40} {'Count':<8} {'F1-Score'}")
print("-"*60)

from sklearn.metrics import f1_score as f1_calc
for disease, count in top_diseases:
    disease_idx = label_encoder.transform([disease])[0]
    disease_mask = y_test == disease_idx
    if disease_mask.sum() > 0:
        disease_f1 = f1_calc(y_test[disease_mask], y_pred[disease_mask], average='micro')
        print(f"{disease:<40} {count:<8} {disease_f1:.3f}")

## 5. Test Prediction with Confidence Score

In [ ]:
# Test prediction example
print("\n🧪 TESTING PREDICTION EXAMPLE:\n")

# Create a sample with a few symptoms
test_symptoms = ['fever', 'cough', 'headache']
test_vector = np.zeros(len(symptom_list))

for symptom in test_symptoms:
    if symptom in symptom_list:
        idx = symptom_list.index(symptom)
        test_vector[idx] = 1

# Reshape for prediction
test_vector = test_vector.reshape(1, -1)

# Make prediction
pred_encoded = model.predict(test_vector)[0]
pred_proba = model.predict_proba(test_vector)[0]
confidence = pred_proba[pred_encoded]

# Decode prediction
predicted_disease = label_encoder.inverse_transform([pred_encoded])[0]

print(f"Input symptoms: {test_symptoms}")
print(f"Predicted disease: {predicted_disease}")
print(f"Confidence score: {confidence:.4f} ({confidence*100:.2f}%)")

# Show top 3 predictions
top_3_idx = np.argsort(pred_proba)[-3:][::-1]
print("\nTop 3 predictions:")
for i, idx in enumerate(top_3_idx, 1):
    disease = label_encoder.inverse_transform([idx])[0]
    prob = pred_proba[idx]
    print(f"  {i}. {disease:<35} {prob:.4f} ({prob*100:.2f}%)")

## 6. Export Model Files

Export three files:
1. `label_encoder.pkl` - For converting numerical predictions to disease names
2. `medicine_model.pkl` - The trained XGBoost model
3. `symptom_list.pkl` - List of symptom names (features)

In [ ]:
print("\n💾 Saving model files...")

# Save label encoder
joblib.dump(label_encoder, 'label_encoder.pkl')
print("✅ Saved: label_encoder.pkl")

# Save model
joblib.dump(model, 'medicine_model.pkl')
print("✅ Saved: medicine_model.pkl")

# Save symptom list
joblib.dump(symptom_list, 'symptom_list.pkl')
print("✅ Saved: symptom_list.pkl")

print("\n✨ All files saved successfully!")
print("\n📥 Download these files and upload them to your disease-service directory:")
print("   1. label_encoder.pkl")
print("   2. medicine_model.pkl")
print("   3. symptom_list.pkl")

## Summary

**Model Performance:**
- The model has been trained using GPU-accelerated XGBoost
- F1-scores (weighted and macro) are displayed above
- Confidence scores are now available for each prediction

**Next Steps:**
1. Download the three `.pkl` files from Colab
2. Upload them to your `disease-service` directory
3. Update your `api.py` to use the new files
4. Test the API with the new confidence scores